# Effects of two knockouts from the same module

Fits, for each of the 1,041 response genes,

```
y ~ K_0 + ... + K_5
```

over cells carrying two knockouts from **one** module, against control cells.

There is no interaction term and there cannot be one: a variable cannot
interact with itself. Each coefficient is therefore a main effect — how far a
same-module double knockout moves the gene, additive and non-additive parts
together. `04` and `05` estimate the within-module interaction instead, by
splitting each module in half.

Writes `ComboEffects_doublesSameGroup.rds` (Figures 5A, 5B) and two tables of
mean expression, of which `ControlCellsMeanExp.rds` becomes the bottom row of
the Figure 5D heatmap.

## Setup

In [ ]:
%load_ext rpy2.ipython

import scanpy as sc
import pandas as pd
import anndata2ri
from rpy2.robjects import numpy2ri, pandas2ri

# The %%R cells below receive pandas frames; these converters are what the
# libraries.py star-import used to activate.
numpy2ri.activate()
pandas2ri.activate()
anndata2ri.activate()

DATASET = "/home/eraslab1/Projects/E3Ligase/analysisSingle/Notebooks/CombinatorialPerturbations/dataset"
MODULES = ["K_0", "K_1", "K_2", "K_3", "K_4", "K_5"]

OUT_FILE = "outputs/ComboEffects_doublesSameGroup.rds"

## Response and design

In [ ]:
adataSingles = sc.read(f"{DATASET}/adataTrainSingles.h5ad")
adataDoubles = sc.read(f"{DATASET}/adataDoubles_sameGroup.h5ad")

adataControlTrain = adataSingles[adataSingles.obs["K_CONTROL"] == 1, ]
adata = sc.AnnData.concatenate(adataDoubles, adataControlTrain)

guideMatrix = adata.obs[MODULES]

expressionMatrix = pd.DataFrame(adata.layers["ClusterResiduals"])
expressionMatrix.columns = adata.var_names
expressionMatrix.index = adata.obs.index

allResp = adata.var_names
my_formula = "y~" + "+".join(MODULES)

print(adata.shape, "|", my_formula)

## One model per gene

:::{note}
The original looped `seq(1, 1042, 1)` over 1,041 genes, so the last iteration
always failed and was swallowed by `tryCatch`. Looping over the actual column
count drops that silent error and leaves the result unchanged.
:::

In [ ]:
%%R -i guideMatrix,expressionMatrix,my_formula,allResp,OUT_FILE
library(broom)

coefDF <- data.frame()

for (i in seq_len(ncol(expressionMatrix))) {
    tryCatch({
        guideMatrix["y"] <- expressionMatrix[, i]
        myFit <- lm(formula(my_formula), data = guideMatrix)
        myDF  <- data.frame(tidy(myFit))
        myDF$respGene <- allResp[i]
        coefDF <- rbind(coefDF, myDF)
    }, error = function(e) message("gene ", i, " skipped: ", conditionMessage(e)))
}

saveRDS(coefDF, OUT_FILE)
dim(coefDF)

## Mean expression per module and in controls

`meanRes_Cont2` is the mean of the cluster residuals, `MeanExp_Cont2` the mean
of the expression itself. Only the control table is read downstream;
`PertCellsMeanExp.rds` is written but nothing in the repository uses it.

In [ ]:
adataSingles = sc.read(f"{DATASET}/adataTrainSingles.h5ad")

# Column suffixes exactly as the original named them: K_0 -> K0, K_CONTROL -> Cont.
SUFFIX = {m: m.replace("_", "") for m in MODULES}
SUFFIX["K_CONTROL"] = "Cont"

columns = {"Genes": adataSingles.var_names}
for module, suffix in SUFFIX.items():
    cells = adataSingles[adataSingles.obs[module] == 1, :]
    columns[f"meanRes_{suffix}"] = cells.layers["ClusterResiduals"].mean(axis=0)
    columns[f"MeanExp_{suffix}"] = cells.X.mean(axis=0)

myTemp = pd.DataFrame(columns)
list(myTemp.columns)

In [ ]:
%%R -i myTemp
saveRDS(myTemp, "outputs/PertCellsMeanExp.rds")
dim(myTemp)

In [ ]:
adataControl = sc.read(f"{DATASET}/adataTestControl.h5ad")

myTemp2 = pd.DataFrame({"Genes": adataControl.var_names,
                        "meanRes_Cont2": adataControl.layers["ClusterResiduals"].mean(axis=0),
                        "MeanExp_Cont2": adataControl.X.mean(axis=0)})
myTemp2.head()

In [ ]:
%%R -i myTemp2
saveRDS(myTemp2, "outputs/ControlCellsMeanExp.rds")
dim(myTemp2)